### question 1 - S&P 500 Stocks Added to the Index

Using the list of S&P 500 companies from Wikipedia's [S&P 500 companies page](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies), download the data including the year each company was added to the index.

Hint: you can use [pandas.read_html](https://pandas.pydata.org/docs/reference/api/pandas.read_html.html) to scrape the data into a DataFrame. You may need to add headers to make it work:
```python
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

# Fetch the HTML content with headers
response = requests.get(url, headers=headers)
```

Steps:
1. Create a DataFrame with company tickers, names, and the year they were added.
2. Extract the year from the addition date and calculate the number of stocks added each year.
3. Which (full) year had the highest number of additions, starting from 2020? 

▎ Note: When stocks are added to the S&P 500, they usually experience a price bump as investors and index funds buy     
  shares following the announcement.

▎ Context: 
> "Following the announcement, all four new entrants saw their stock prices rise in extended trading on Friday" - recent examples of S&P 500 additions include DASH, WSM, EXE, TKO in 2025 ([Nasdaq article](https://www.nasdaq.com/articles/sp-500-reshuffle-dash-tko-expe-wsm-join-worth-buying)).

*Additional*: How many current S&P 500 stocks have been in the index for more than 20 years? 

In [5]:
import pandas as pd
import requests
from io import StringIO

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)

# Wrap the raw HTML string in StringIO
html_data = StringIO(response.text)
df = pd.read_html(html_data)[0]

# Clean the date column and extract the year
df['Date added'] = pd.to_datetime(df['Date added'].astype(str).str.slice(0, 10), errors='coerce')
df['Year Added'] = df['Date added'].dt.year

# Count additions from 2020 onward
additions_since_2020 = df[df['Year Added'] >= 2020]['Year Added'].value_counts()
print(additions_since_2020)

Year Added
2025    18
2024    16
2023    15
2022    15
2026    13
2021    10
2020    10
Name: count, dtype: int64


### question 2 - [Macro] Indexes YTD (as of 21 August 2026)

**How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?**

Using Yahoo Finance World Indices data, compare the year-to-date (YTD) performance (1 January-21 August 2026, use the closing price to calculate the growth) of major stock market indexes for the following countries:
* United States - S&P 500 (^GSPC)
* China - Shanghai Composite (000001.SS)
* Hong Kong - HANG SENG INDEX (^HSI)	
* Australia - S&P/ASX 200 (^AXJO)
* India - Nifty 50 (^NSEI)
* Canada - S&P/TSX Composite (^GSPTSE)
* Germany - DAX (^GDAXI)
* United Kingdom - FTSE 100 (^FTSE)
* Japan - Nikkei 225 (^N225)
* Mexico - IPC Mexico (^MXX)
* Brazil - Ibovespa (^BVSP)

*Hint*: use ```start_date='2026-01-01'``` and ```end_date='2026-08-21'``` parameters when downloading daily data in yfinance

Context: 
> [Global Valuations: Who's Cheap, Who's Not?](https://simplywall.st/article/beyond-the-us-global-markets-after-yet-another-tariff-update) article suggests "Other regions may be growing faster than the US and you need to diversify."

Reference: Yahoo Finance World Indices - https://finance.yahoo.com/world-indices/

*Additional*: How many of these indexes have better returns than the S&P 500 over 3, 5, and 10 year periods? Do you see the same trend?
Note: For simplicity, ignore currency conversion effects.

In [9]:
import yfinance as yf

tickers = ['^GSPC', '000001.SS', '^HSI', '^AXJO', '^NSEI', '^GSPTSE', '^GDAXI', '^FTSE', '^N225', '^MXX', '^BVSP']
data = yf.download(tickers, start='2026-01-01', end='2026-08-22')['Close']

# Fill missing values caused by differing global market holidays
data = data.bfill().ffill()

# Calculate YTD returns
ytd_returns = (data.iloc[-1] / data.iloc[0]) - 1
sp500_return = ytd_returns['^GSPC']

# Filter and count
outperformers = ytd_returns[ytd_returns > sp500_return]
print(outperformers)
print(f"Indexes beating S&P 500: {len(outperformers)}")

[*********************100%***********************]  11 of 11 completed

Ticker
^GSPTSE    0.148566
^N225      0.273641
dtype: float64
Indexes beating S&P 500: 2


### Question 3 - [Index] S&P 500 Market Corrections Analysis


**Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.**

For this task, define a correction as an event when a stock index goes down by **at least 5%** from the most recent all-time high.

Steps:
1. Download S&P 500 historical data (period from 1950 to present) using yfinance (daily stats)
2. Identify all-time high closing price points (where price exceeds all previous days' prices)
3. For each pair of consecutive all-time highs, find the minimum price in between
4. Calculate drawdown percentages: (high - low) / high × 100
5. Filter for corrections with at least 5% drawdown
6. Calculate the duration in days for each correction period
7. Determine the 25th, 50th (median), and 75th percentiles for correction durations and drawdowns

*Context:* 
> * Investors often wonder about the typical length of market corrections when deciding "when to buy the dip" ([Reddit discussion](https://www.reddit.com/r/investing/comments/1jrqnte/when_are_you_buying_the_dip/?rdt=64135)).

> * [A Wealth of Common Sense - How Often Should You Expect a Stock Market Correction?](https://awealthofcommonsense.com/2022/01/how-often-should-you-expect-a-stock-market-correction/)

*Hint (use this data to compare with your results)*: Here is the list of top 10 largest corrections by drawdown:
* 2007-10-09 to 2009-03-09: 56.8% drawdown over 517 days
* 2000-03-24 to 2002-10-09: 49.1% drawdown over 929 days
* 1973-01-11 to 1974-10-03: 48.2% drawdown over 630 days
* 1968-11-29 to 1970-05-26: 36.1% drawdown over 543 days
* 2020-02-19 to 2020-03-23: 33.9% drawdown over 33 days
* 1987-08-25 to 1987-12-04: 33.5% drawdown over 101 days
* 1961-12-12 to 1962-06-26: 28.0% drawdown over 196 days
* 1980-11-28 to 1982-08-12: 27.1% drawdown over 622 days
* 2022-01-03 to 2022-10-12: 25.4% drawdown over 282 days
* 1966-02-09 to 1966-10-07: 22.2% drawdown over 240 days

In [11]:
import yfinance as yf
import pandas as pd

# 1. Download daily S&P 500 historical data (using double brackets to extract as DataFrame)
df = yf.download('^GSPC', start='1950-01-01')[['Close']]
df.columns = ['Close'] 

# 2. Identify the running all-time high (ATH) for every day
df['ATH'] = df['Close'].cummax()

# 3. Group by each distinct ATH to find the specific correction periods
peak_dates = df.reset_index().groupby('ATH')['Date'].min().reset_index()
peak_dates.columns = ['ATH', 'Peak_Date']

# Find the minimum closing price and its corresponding date (Trough_Date) for each ATH period
corrections = df.reset_index().groupby('ATH').apply(
    lambda x: pd.Series({
        'Low': x['Close'].min(),
        'Trough_Date': x.loc[x['Close'].idxmin(), 'Date']
    })
).reset_index()

# Merge peak dates with trough data
corrections = corrections.merge(peak_dates, on='ATH')

# 4. Calculate drawdown percentages
corrections['Drawdown'] = (corrections['ATH'] - corrections['Low']) / corrections['ATH'] * 100

# 5. Filter for significant corrections (at least 5% drawdown)
significant_corrections = corrections[corrections['Drawdown'] >= 5.0].copy()

# 6. Calculate the duration in days from Peak to Trough
significant_corrections['Duration'] = (significant_corrections['Trough_Date'] - significant_corrections['Peak_Date']).dt.days

# Sort by largest drawdown to compare with the course hint
top_10 = significant_corrections.sort_values(by='Drawdown', ascending=False).head(10)
print("--- Top 10 Largest Corrections ---")
print(top_10[['Peak_Date', 'Trough_Date', 'Drawdown', 'Duration']].to_string(index=False))

# 7. Determine percentiles for durations and drawdowns
percentiles = significant_corrections[['Drawdown', 'Duration']].describe(percentiles=[.25, .5, .75])
print("\n--- Percentiles ---")
print(percentiles.loc[['25%', '50%', '75%']])

[*********************100%***********************]  1 of 1 completed


--- Top 10 Largest Corrections ---
 Peak_Date Trough_Date  Drawdown  Duration
2007-10-09  2009-03-09 56.775388       517
2000-03-24  2002-10-09 49.146948       929
1973-01-11  1974-10-03 48.203593       630
1968-11-29  1970-05-26 36.061641       543
2020-02-19  2020-03-23 33.924960        33
1987-08-25  1987-12-04 33.509515       101
1961-12-12  1962-06-26 27.973568       196
1980-11-28  1982-08-12 27.113582       622
2022-01-03  2022-10-12 25.425097       282
1966-02-09  1966-10-07 22.177335       240

--- Percentiles ---
      Drawdown  Duration
25%   6.234677     22.00
50%   7.986358     40.50
75%  14.019826     86.25


### Question 4 - [Stocks] Earnings Surprise Analysis for Amazon (AMZN)


**Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.**

Steps:
1. Load earnings data for the last few years using the yfinance method get_earnings_dates():
```python
import yfinance as yf

ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)
result = ticker_obj.get_earnings_dates()
```
You should have 25 entries starting from 2020-10-29. Note: the 1 future earnings date entry will not have Reported EPS  
and Surprise % filled in.

2. Download complete historical price data using yfinance
3. Calculate 2-day percentage changes for all historical dates: for each sequence of 3 consecutive trading days (Day 1, Day 2, Day 3), compute the *return* as Close_Day3 / Close_Day1 - 1. (Assume Day 2 may correspond to the earnings announcement.)
4. Filter for positive earnings surprises and calculate the median 2-day return. Then calculate the correlation between
   the 2-day stock return and the earnings surprise magnitude.

▎ *Hint*: you can generate all correlations at once using pd.corr().

 > **Context:** Earnings announcements, especially when they exceed analyst expectations, can significantly impact stock prices in the short term.

**Reference:** [Yahoo Finance Earnings Calendar](https://finance.yahoo.com/calendar/earnings?symbol=AMZN)

*Additional*: Is there a correlation between the magnitude of the earnings surprise and the stock price reaction? Does the market react differently to earnings surprises during bull vs. bear markets?


In [13]:
import yfinance as yf
import pandas as pd

# 1. Load AMZN earnings dates
ticker = 'AMZN'
amzn = yf.Ticker(ticker)
earnings = amzn.get_earnings_dates()

# Clean the timezone from the earnings dates to allow proper merging
earnings = earnings.tz_localize(None).reset_index()
earnings['Earnings Date'] = earnings['Earnings Date'].dt.date

# Dynamically locate the exact name of the surprise column 
surprise_col = [col for col in earnings.columns if 'Surprise' in col][0]

# 2. Download historical price data
history = amzn.history(period="10y").reset_index()
history['Date'] = history['Date'].dt.tz_localize(None).dt.date

# 3. Calculate 2-day percentage changes (Close_Day3 / Close_Day1 - 1)
history['Return_2d'] = history['Close'].shift(-1) / history['Close'].shift(1) - 1

# 4. Filter for positive earnings surprises using the dynamic column name
positive_earnings = earnings[earnings[surprise_col] > 0].copy()

# 5. Merge datasets on the earnings date
merged = pd.merge(positive_earnings, history[['Date', 'Return_2d']], left_on='Earnings Date', right_on='Date', how='inner')

# Calculate median return and correlation
median_return = merged['Return_2d'].median()
correlation = merged['Return_2d'].corr(merged[surprise_col])

print(f"Median 2-day return for positive surprises: {median_return:.4%}")
print(f"Correlation between surprise magnitude and return: {correlation:.4f}")

Median 2-day return for positive surprises: 0.3528%
Correlation between surprise magnitude and return: 0.3367
